In [1]:
from pathlib import Path  
from massipipe.pipeline import PipelineProcessor, find_datasets
from massipipe.utils import rgb_subset_from_hsi, read_envi, percentile_stretch_image, save_png
from massipipe.mosaic import convert_geotiff_to_8bit
from massipipe.quicklook import QuickLookProcessor
from rich import print

In [2]:
# Parameters
dataset_dir = Path('/home/mha114/data/seabee-minio/larvik/kongsbakkebukta/aerial/hsi/20230830/massimal_larvik_kongsbakkebukta_202308301328_hsi')
output_base_dir = Path('/home/mha114/data/export')

rgb_wl_rededge = (700,550,460)

In [3]:
pipeline = PipelineProcessor(dataset_dir)

09:58:56 INFO: ----------------------------------------------------------------
09:58:56 INFO: File logging for massimal_larvik_kongsbakkebukta_202308301328_hsi initialized.
09:58:56 INFO: Found directory 0_raw - processing from raw files.


In [4]:
pipeline.config.radiance_gc.overwrite = True
pipeline.config.radiance_gc.smooth_spectra = True
pipeline.config.radiance_gc_rgb.overwrite = True
pipeline.config.general.rgb_wl = rgb_wl_rededge

print(pipeline.config)

MassipipeOptions(
    general=MpGeneral(rgb_wl=(700, 550, 460)),
    quicklook=MpQuickLook(create=True, overwrite=False, percentiles=None),
    imu_data=MpImuData(create=True, overwrite=False),
    geotransform=MpGeoTransform(
        create=True,
        overwrite=True,
        camera_opening_angle_deg=36.5,
        pitch_offset_deg=0.0,
        roll_offset_deg=0.0,
        altitude_offset_m=0.0,
        utm_x_offset_m=0.0,
        utm_y_offset_m=0.0,
        assume_square_pixels=True
    ),
    radiance=MpRadiance(
        create=True,
        overwrite=False,
        set_saturated_pixels_to_zero=True,
        add_envi_mapinfo_to_header=True,
        add_irradiance_to_header=True
    ),
    radiance_rgb=MpRadianceRgb(create=True, overwrite=False),
    radiance_gc=MpRadianceGc(
        create=True,
        overwrite=True,
        smooth_spectra=True,
        subtract_dark_spec=False,
        set_negative_values_to_zero=False,
        reference_image_numbers=[14, 16, 17, 22],
        reference_image_ranges=[
            (1230, 1420, 0, 900),
            (340, 640, 0, 900),
            (1120, 1520, 0, 900),
            (1400, 2000, 0, 900)
        ]
    ),
    radiance_gc_rgb=MpRadianceGcRgb(create=True, overwrite=True),
    irradiance=MpIrradiance(create=True, overwrite=False),
    reflectance=MpReflectance(
        create=True,
        overwrite=False,
        wl_min=400.0,
        wl_max=930.0,
        conv_irrad_with_gauss=True,
        fwhm_irrad_smooth=3.5,
        smooth_spectra=False,
        refl_from_mean_irrad=False
    ),
    reflectance_gc=MpReflectanceGc(create=True, overwrite=False, smooth_spectra=True, method='from_rad_gc'),
    reflectance_gc_rgb=MpReflectanceGcRgb(create=True, overwrite=False),
    mosaic=MpMosaic(
        overview_factors=[2, 4, 8, 16, 32],
        radiance_gc_rgb=MpMosaicCreateOverwrite(create=True, overwrite=True),
        reflectance_gc_rgb=MpMosaicCreateOverwrite(create=False, overwrite=True)
    )
)

In [5]:
# pipeline.glint_correct_radiance_images()

In [5]:
pipeline.create_glint_corrected_radiance_rgb_geotiff()

09:59:40 INFO: ---- EXPORTING RGB GEOTIFF FOR GLINT CORRECTED RADIANCE ----
09:59:40 INFO: Exporting RGB GeoTIFF from massimal_larvik_kongsbakkebukta_202308301328_hsi_000_radiance_gc.bip.hdr.
09:59:43 INFO: Exporting RGB GeoTIFF from massimal_larvik_kongsbakkebukta_202308301328_hsi_001_radiance_gc.bip.hdr.
09:59:46 INFO: Exporting RGB GeoTIFF from massimal_larvik_kongsbakkebukta_202308301328_hsi_002_radiance_gc.bip.hdr.
09:59:50 INFO: Exporting RGB GeoTIFF from massimal_larvik_kongsbakkebukta_202308301328_hsi_003_radiance_gc.bip.hdr.
09:59:52 INFO: Exporting RGB GeoTIFF from massimal_larvik_kongsbakkebukta_202308301328_hsi_004_radiance_gc.bip.hdr.
09:59:55 INFO: Exporting RGB GeoTIFF from massimal_larvik_kongsbakkebukta_202308301328_hsi_005_radiance_gc.bip.hdr.
09:59:58 INFO: Exporting RGB GeoTIFF from massimal_larvik_kongsbakkebukta_202308301328_hsi_006_radiance_gc.bip.hdr.
10:00:02 INFO: Exporting RGB GeoTIFF from massimal_larvik_kongsbakkebukta_202308301328_hsi_007_radiance_gc.bip.h

In [6]:
# Export PNGs and 8-bit GeoTiffs
output_dir = output_base_dir / pipeline.dataset_base_name
output_dir.mkdir(exist_ok=True)

for rad_gc_im, rad_gc_im_rgb in zip(pipeline.rad_gc_im_paths, pipeline.rad_gc_rgb_im_paths):
    image,wl,metadata = read_envi(rad_gc_im)
    
    # Create percentile stretched version for PNG
    image[image<0] = 0
    image_rgb, rgb_wl = rgb_subset_from_hsi(image,wl,rgb_target_wl=rgb_wl_rededge)
    image_rgb = percentile_stretch_image(image_rgb)
    
    png_path = output_dir / (rad_gc_im.name.split(".")[0] + '.png')    
    save_png(image_rgb,png_path)
    
    # Convert existing GeoTIFF to 8-bit
    geotiff_8bit_path = output_dir / rad_gc_im_rgb.name
    convert_geotiff_to_8bit(rad_gc_im_rgb,geotiff_8bit_path)


10:01:00 INFO: Converting massimal_larvik_kongsbakkebukta_202308301328_hsi_000_radiance_gc_rgb.tiff to 8-bit geotiff
10:01:04 INFO: Converting massimal_larvik_kongsbakkebukta_202308301328_hsi_001_radiance_gc_rgb.tiff to 8-bit geotiff
10:01:07 INFO: Converting massimal_larvik_kongsbakkebukta_202308301328_hsi_002_radiance_gc_rgb.tiff to 8-bit geotiff
/home/mha114/code/github/massipipe/massipipe/utils.py:406: RuntimeWarning: divide by zero encountered in scalar divide
  image_stretched[:, :, band_index] = (image_band - p_low) * (255 / p_range)
/home/mha114/code/github/massipipe/massipipe/utils.py:406: RuntimeWarning: invalid value encountered in multiply
  image_stretched[:, :, band_index] = (image_band - p_low) * (255 / p_range)
/home/mha114/code/github/massipipe/massipipe/utils.py:409: RuntimeWarning: invalid value encountered in cast
  return image_stretched.astype(np.uint8)
10:01:09 INFO: Converting massimal_larvik_kongsbakkebukta_202308301328_hsi_003_radiance_gc_rgb.tiff to 8-bit geo

FileNotFoundError: Unable to locate file "/home/mha114/data/seabee-minio/larvik/kongsbakkebukta/aerial/hsi/20230830/massimal_larvik_kongsbakkebukta_202308301328_hsi/1b_radiance_gc/massimal_larvik_kongsbakkebukta_202308301328_hsi_025_radiance_gc.bip.hdr". If the file exists, use its full path or place its directory in the SPECTRAL_DATA environment variable.